# Nereus from Python — a 40-minute tour

Nereus fits exoplanet orbits. What makes it different from most orbit codes is that it
samples **over the model**, not just within one:

| it samples over | meaning |
|---|---|
| **how many companions** | the trans-dimensional problem — P(N planets \| data), not a fixed N |
| **which noise description** | a menu of correlated-noise models competing under exclusion groups, so two descriptions of the same stellar signal cannot both be active |
| **which observable constrains which companion** | per planet. A companion the RV nails can be astrometrically undetected — forcing the coupling makes every fit report an inclination, and therefore a "dynamical mass", whether or not the astrometry constrained one |

Eighteen samplers share one interface (parallel tempering, nested sampling, RJMCMC,
MoMS, Daedalus, SMC, MAP, OFTI, …) with thermodynamic-integration, stepping-stone and
bridge evidence estimators.

**Observables**: radial velocities, transits, absolute and relative astrometry,
Rossiter–McLaughlin, Doppler tomography, transit timing — under one likelihood.

You do not need Julia installed. `astronereus` fetches a prebuilt runtime on first use
and runs it out-of-process.

## 0. Setup

The first call downloads a ~480 MB runtime and compiles it once (several minutes,
with a progress bar). Every session after that starts in ~20 s.

**Do this the day before a workshop, not during it.**

In [ ]:
# !pip install astronereus numpy
import numpy as np, astronereus
from astronereus import engines

print(astronereus.__version__)
print(astronereus.runtime_parts())   # what is installed

In [ ]:
# One daemon, reused by every fit below. Do NOT use `with` in a notebook:
# the daemon would die at the end of the cell and the next one pays startup again.
s = astronereus.session().start()
s.ping()

---
# 1. Gaia-4 b — an orbit from astrometry alone

Gaia DR4 epoch astrometry: 824 along-scan abscissae over 4.94 yr. No radial
velocities at all. The along-scan residual before any orbit model is **1031 σ** —
this is not a marginal detection.

The data is *named*, not uploaded: Nereus fetches the DR4 VOTable itself.

In [ ]:
GAIA4 = {"catalogue": "gaia_dr4", "source_id": 1457486023639239296}

# plx and M_pri are PRIORS, so they go in the priors dict -- there is no
# parallax= or m_pri= keyword. The parallax prior must be informative: the
# abscissae constrain a0 ~ M_sec * plx, degenerate without it.
res_g4 = s.fit_astrometry(
    iad     = [GAIA4],
    planets = 1,
    priors  = {
        "plx":      {"type": "normal",     "mu": 13.628, "sigma": 0.021},  # mas
        "M_pri":    {"type": "fixed",      "value": 0.644},   # M_sun, Stefansson+ 2024
        "a_k1":     {"type": "loguniform", "lo": 0.3,   "hi": 4.0},    # AU
        "M_sec_k1": {"type": "loguniform", "lo": 0.001, "hi": 0.05},   # M_sun
        "inc_k1":   {"type": "sine"},
    },
    engine = engines.PT(n_rounds=4, n_chains=8, seed=42),  # 12 for production
)
res_g4

In [ ]:
# Published (Stefansson+ 2024): P = 571.3 d, M = 11.8 M_Jup
p = res_g4.params
for k in ("a_k1", "M_sec_k1", "inc_k1"):
    if k in p: print(f"  {k:10} {p[k]}")

---
# 2. HD 114762 — the same system, three ways

HD 114762 b was announced in 1989 as the first exoplanet candidate, with
M sin i ≈ 11 M_Jup — right at the deuterium-burning line. Whether it is a planet
has depended for 35 years on one unknown: the inclination.

It has all three data types:

| channel | what |
|---|---|
| RV | 24 Keck/HIRES + 35 Lick over 29 yr (Rosenthal+ 2021) |
| Hipparcos | HIP 64426 intermediate astrometric data |
| Gaia DR4 | 558 abscissae, pre-fit residual 10.3 σ |

We fit it three times to show what each channel can and cannot do.

In [ ]:
import urllib.request

URL = "https://raw.githubusercontent.com/jvines/Nereus.jl/main/test/data/hd114762_rv.dat"
tb = {}
for line in urllib.request.urlopen(URL).read().decode().splitlines():
    t = line.strip()
    if not t or t.startswith("#"):
        continue
    jd, rv, er, ins = t.split()
    d = tb.setdefault(ins, {"t": [], "rv": [], "rv_err": []})
    d["t"].append(float(jd) - 2_400_000.5)      # JD -> MJD
    d["rv"].append(float(rv)); d["rv_err"].append(float(er))

# HIRES and Lick stay SEPARATE channels: different zero points, different jitter.
rv_ch = astronereus.RV(data=tb)
{k: len(v["t"]) for k, v in tb.items()}

In [ ]:
HIP = {"catalogue": "hipparcos", "hip": 64426}
DR4 = {"catalogue": "gaia_dr4",  "source_id": 3937211745905473024}

# P = 83.92 d has been known since 1989. Bracket it; do not re-search it.
#
# Two dicts because the PARAMETRISATION differs. RV-only is K-driven: its free
# parameters are P and K. There is no `a` and no `M_sec`, because RV alone
# cannot measure a semi-major axis or a true mass -- only K = f(M_sec sin i).
# Those parameters exist only once astrometry is in the fit.
PRIORS_RV = {"P_k1": {"type": "loguniform", "lo": 60.0, "hi": 105.0}}   # days

PRIORS_AS = {
    "plx":      {"type": "normal",     "mu": 25.36, "sigma": 0.30},  # Gaia DR3
    "M_pri":    {"type": "fixed",      "value": 0.83},   # M_sun, Kiefer+ 2019
    "a_k1":     {"type": "loguniform", "lo": 0.30,  "hi": 0.45},   # AU
    "M_sec_k1": {"type": "loguniform", "lo": 0.003, "hi": 0.5},    # reaches STELLAR
}
ENGINE = engines.PT(n_rounds=6, n_chains=8, seed=42)

## 2a. RV only — gives M sin i, and nothing about i

This is the 1989 measurement, redone. It cannot distinguish an 11 M_Jup planet
seen edge-on from a 0.1 M_sun star seen nearly face-on.

In [ ]:
res_rv = s.fit_rv(rv_ch, planets=1, priors=PRIORS_RV, engine=ENGINE)
res_rv

## 2b. Astrometry only — Hipparcos + Gaia DR4 together

Two missions, ~25 yr apart, as **two numbered instruments** in one fit. The DR4
per-CCD abscissa model *is* the Hipparcos IAD model — same container, same
likelihood — so combining them is a list, not a special case.

Astrometry sees the photocentre wobble, so it constrains the *true* mass and the
inclination — but on its own it is far weaker on the period than 29 yr of RV.

In [ ]:
res_as = s.fit_astrometry(
    iad     = [HIP, DR4],     # order is the instrument numbering
    planets = 1,
    priors  = PRIORS_AS,
    engine  = ENGINE,
)
res_as

## 2c. Joint — RV **and** astrometry

The RV pins the period and eccentricity; the astrometry breaks sin i. Neither
channel gives a true mass alone; together they do.

This is the whole argument for joint fitting, on the system where it has mattered
longest.

In [ ]:
as_ch = astronereus.Astrometry(iad=[HIP, DR4])

res_joint = s.fit_joint(rv_ch, as_ch, planets=1, priors=PRIORS_AS,
                        engine=ENGINE, output_dir="hd114762_joint")
res_joint

## 2d. What each channel bought

In [ ]:
def show(tag, res):
    p = res.params or {}
    # RV-only is K-driven, so it has P/K; the astrometric fits have a/M_sec/inc.
    keys = [k for k in ("P_k1", "K_k1", "a_k1", "M_sec_k1", "inc_k1") if k in p]
    print(f"{tag:14} logZ={res.log_z}")
    for k in keys:
        print(f"    {k:10} {p[k]}")

show("RV only",     res_rv)      # P, K  -> M sin i only
show("Astrom only", res_as)      # a, M_sec, inc
show("Joint",       res_joint)   # the one with a true mass

In [ ]:
# Figures are written under output_dir
from IPython.display import Image
import glob
for f in sorted(glob.glob("hd114762_joint/**/*.png", recursive=True))[:3]:
    print(f); display(Image(f))

---
## Where to go next

- **Trans-dimensional**: `planets=range(0, 4)` samples over the planet count and
  returns P(N | data) instead of assuming one.
- **Noise menu**: `default_noise_menu` puts correlated-noise models in competition
  under exclusion groups.
- **Other engines**: `engines.Nested`, `engines.Daedalus` (trans-dim posterior *and*
  evidence in one run), `engines.TransdimPTEmcee` for blind search.
- `run_job(cfg)` is the batch entry point — same machinery, a JSON/dict schema.

Raise `n_rounds` to 12+ for anything you would put in a paper. The settings here
are sized to finish while you watch.

In [ ]:
s.stop()   # or leave it; it idles out after 30 min